# 異種モデルによる分類制約付きベイズ最適化

回帰GPで目的値を予測し、分類GPで「許容クラスに属する確率」を予測して、次の実験候補を選びます。**分類確率による重み付けはハードな安全保証ではありません**。

[理論](../../docs/theory/30_heterogeneous_classification_constraints.md) / [実装ガイド](../../docs/guide/30_heterogeneous_classification_constraints.md)

## 1. 合成データとモデル

連続目的値 `score` を最大化し、分類ラベル `status` が許容集合に属する確率を用います。ここでは二値分類で動作を確認します。多クラス・順序分類も、分類器が `predict_proba` でクラス確率を返す場合は同じクラス集合の考え方です。

In [ ]:
import torch
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.acquisition import make_classification_constrained_qei_acquisition
from robotorchan.models.classification.binary.standard.single_task import BinarySingleTaskGPClassifier
from robotorchan.models.heterogeneous import HeterogeneousModel
from robotorchan.models.standard.single_task import SingleTaskGP
from robotorchan.optim import optimize_acqf
from robotorchan.semantics import ClassificationConstraint, ProblemSemantics, RegressionObjective

torch.manual_seed(11)
train_X = torch.linspace(0, 1, 12, dtype=torch.double).unsqueeze(-1)
train_Y = 1 - (train_X - 0.7).square()
train_C = (train_X.squeeze(-1) > 0.4).long()
regression = SingleTaskGP(train_X, train_Y)
classifier = BinarySingleTaskGPClassifier(train_X, train_C)
model = HeterogeneousModel(regression, classifier, output_names=["score", "status"])

## 2. 許容クラスと獲得関数

`feasible_classes=(1,)` はクラス1のみ許容します。複数のクラスを許容する場合は `(0, 2)` のように指定します（そのクラスが実在する分類器が必要です）。`best_f` は観測された回帰目的の最良値です。

分類確率を `p(x)` とすると、q=1では `qEI(x) × p(x)` を評価します。

In [ ]:
semantics = ProblemSemantics(
    objectives=(RegressionObjective("score"),),
    constraints=(ClassificationConstraint("status", feasible_classes=(1,)),),
)
acquisition = make_classification_constrained_qei_acquisition(
    model,
    semantics,
    best_f=train_Y.max().item(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64]), seed=11),
)
grid = torch.linspace(0, 1, 51, dtype=torch.double).reshape(-1, 1, 1)
with torch.no_grad():
    scores = acquisition(grid)
print("獲得関数の出力形状:", tuple(scores.shape))
print("最大スコア:", scores.max().item())

## 3. 候補提案まで実行

既存の `optimize_acqf` を使います。ここでは再現性と実行の簡単さのためSobol候補探索を使い、候補が境界内にあることを確認します。

In [ ]:
bounds = torch.tensor([[0.0], [1.0]], dtype=torch.double)
candidate, value = optimize_acqf(
    acquisition, bounds, q=1, optimizer="sobol", raw_samples=256, seed=11
)
print("次の実験候補:", candidate)
print("獲得関数値:", value)
print("候補の許容クラス確率:", classifier.predict_proba(candidate)[..., 1])
assert candidate.shape == (1, 1)
assert torch.all(candidate >= bounds[0]) and torch.all(candidate <= bounds[1])

## 4. 解釈と限界

- q>1では分類器の各候補の確率を掛ける近似であり、候補間の相関を考慮した同時実行可能確率ではありません。
- `probability_threshold=0.8` のような閾値制約は、この獲得関数では扱えません。
- Mixed変数では `MixedVariableSpace` と対応Optimizerを使いますが、モデルがその変数を適切に学習できるかは別問題です。
- この例は候補提案までです。実際の実験後には新しい回帰値と分類ラベルを観測し、モデルを更新して次の反復へ進みます。

参考文献は[理論章](../../docs/theory/30_heterogeneous_classification_constraints.md)を参照してください。